# DINOv3 SAT sur HEALPix — réglage sur une seule date

Ce notebook fait tourner `GetDINOV3SAT` sur **une date** du store Sentinel-2 HEALPix
et donne de quoi comprendre ce que fait le réseau avant de faire confiance à une
classification.

L'ordre des cellules est fait pour qu'on **ne recalcule pas ce qui coûte cher** :

| étape | coût | cellule à rejouer |
|---|---|---|
| lecture de la date | ~1 min la 1re fois, puis cache disque | §2 |
| découpage en tuiles | secondes | §3 |
| passage dans DINOv3 | 10 s – 2 min | §4 |
| PCA des tokens | instantané | §5 |
| **k-means** | instantané | **§6 — c'est ici qu'on règle `K`** |
| cartes / UMAP | secondes | §7, §8 |

Donc : on exécute §1 → §4 une fois, puis on boucle sur §6 en changeant `K`.
Si on change `TILE_LEVELS` ou `MIN_COVERAGE`, il faut reprendre à §3.

## 1. Paramètres

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # pour importer le script voisin
sys.path.insert(0, str(Path.cwd().parent))   # pour importer healpix_analyse en dev

# --------------------------------------------------------------------------
# À RÉGLER
# --------------------------------------------------------------------------
ZARR    = "https://data-taos.ifremer.fr/EGU25_CFOSAT/Sentinel2_test.zarr"
CACHE   = os.path.expanduser("~/s2_cache")   # None pour ne rien mettre en cache
WEIGHTS = os.path.expanduser("~/QTRACE/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth")

TIME_INDEX   = 0      # la date étudiée (voir la liste imprimée en §2)
TILE_LEVELS  = 8      # côté des tuiles DINO = 2**TILE_LEVELS px (8 -> 256 px)
MIN_COVERAGE = 1.0    # 1.0 = uniquement les tuiles parent COMPLÈTES (aucun NaN)
K            = 12     # nombre de groupes k-means (réglé en §6)

FAKE = False          # True = réseau aléatoire, pour tester la mécanique sans poids
# --------------------------------------------------------------------------

from healpix_analyse.dino import GetDINOV3SAT, load_dinov3_sat, nested_to_tiles
from dino_umap_sentinel2 import (
    FakeDino, RGB, categorical_cmap, cell_dim, healpix_level, open_store, rgb_at,
)

print("bandes RGB utilisées :", RGB)

## 2. Lecture d'une date

La première exécution télécharge le chunk de la date (quelques centaines de Mo) ;
avec `CACHE`, les suivantes le relisent sur disque.

In [ ]:
ds    = open_store(ZARR)
level = healpix_level(ds)
cell_id = ds["cell_ids"].values.astype(np.int64)
dates = [str(d)[:10] for d in ds["time"].values]

print(f"niveau HEALPix {level}, {cell_id.size} cellules, {len(dates)} dates")
print("dates :", ", ".join(f"{i}:{d}" for i, d in enumerate(dates[:12])), "...")

t0 = time.time()
rgb = rgb_at(ds, TIME_INDEX, cache=CACHE)          # [N, 3] réflectances dans [0, 1]
print(f"date {TIME_INDEX} = {dates[TIME_INDEX]} lue en {time.time()-t0:.0f}s ; "
      f"NaN : {100*np.isnan(rgb).any(1).mean():.1f}% des cellules")

## 3. Découpage en tuiles — *ce que le réseau va vraiment voir*

C'est l'étape à regarder en premier quand un résultat semble incompréhensible.
`nested_to_tiles` replie chaque cellule de `parent_level` en une image carrée
exacte (aucun rééchantillonnage). Avec `fill="nan"` on voit les trous tels quels.

Rappel d'orientation : le nord d'une face HEALPix pointe vers le **coin
haut-droit** de l'image, pas vers le haut. Une image qui « penche » de 45° est
normale.

In [ ]:
parent_level = level - TILE_LEVELS
tiles, parent_ids, valid, coverage = nested_to_tiles(
    rgb, cell_id, level, parent_level, fill="nan")

keep = coverage >= MIN_COVERAGE
print(f"{tiles.shape[0]} tuiles de {tiles.shape[-1]} px au niveau {parent_level} ; "
      f"{keep.sum()} complètes à >= {MIN_COVERAGE}")
print("couverture : min %.3f  médiane %.3f  max %.3f"
      % (coverage.min(), np.median(coverage), coverage.max()))

fig, ax = plt.subplots(figsize=(6, 2.4))
ax.hist(coverage, bins=40, color="0.4")
ax.axvline(MIN_COVERAGE, color="crimson", lw=2, label=f"MIN_COVERAGE = {MIN_COVERAGE}")
ax.set_xlabel("fraction de pixels utilisables par tuile"); ax.set_ylabel("tuiles")
ax.legend(); plt.show()

In [ ]:
def show_tiles(imgs, titles=None, ncol=6, size=2.1, cmap=None, **kw):
    """Planche de vignettes."""
    n = len(imgs)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow), squeeze=False)
    for i, a in enumerate(axes.ravel()):
        a.set_axis_off()
        if i >= n:
            continue
        a.imshow(imgs[i], cmap=cmap, **kw)
        if titles is not None:
            a.set_title(titles[i], fontsize=8)
    fig.tight_layout()
    return fig

def to_rgb(tile, p_low=2, p_high=98):
    """[3, S, S] -> image affichable, étirée sur des percentiles (NaN en noir)."""
    img = np.transpose(tile, (1, 2, 0))
    lo, hi = np.nanpercentile(img, [p_low, p_high])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)

idx = np.where(keep)[0][:12]
show_tiles([to_rgb(tiles[i]) for i in idx],
           [f"{parent_ids[i]}  cov={coverage[i]:.3f}" for i in idx])
plt.show()

Si ces vignettes ne ressemblent pas à des paysages, inutile d'aller plus loin :
le problème est en amont (bandes, échelle des réflectances, niveau HEALPix).
Comparer avec `TILE_LEVELS` plus grand si les tuiles sont trop petites pour
montrer une structure.

## 4. Passage dans DINOv3

`return_patches=True` donne un vecteur par cellule de `level - 4`
(un patch de 16 px), c'est ce qui sert à la classification.

In [ ]:
model = FakeDino() if FAKE else load_dinov3_sat("dinov3_vitl16", WEIGHTS)

t0 = time.time()
res = GetDINOV3SAT(
    rgb, cell_id, level, parent_level,
    model=model, return_patches=True, min_coverage=MIN_COVERAGE,
)
G = int(round(np.sqrt(res.patch_embedding.shape[0] / max(res.cell_id.size, 1))))
E = res.patch_embedding.reshape(res.cell_id.size, G, G, -1)     # [tuile, ligne, colonne, D]

print(f"{res.cell_id.size} tuiles -> {res.patch_embedding.shape} en {time.time()-t0:.0f}s")
print(f"grille de patches : {G}x{G} par tuile, dimension {E.shape[-1]}, "
      f"niveau des patches {res.patch_level}")

# les tuiles gardées, dans le même ordre que res.cell_id
sel = np.searchsorted(parent_ids, res.cell_id)
kept_tiles = tiles[sel]

## 5. Les features ont-elles un sens ? — PCA des tokens

Le test standard sur DINO : projeter les tokens de patch sur leurs **3 premières
composantes principales** et les afficher en RGB. Si le réseau « voit » quelque
chose, cette image fait apparaître les objets (champs, forêt, routes, eau) sans
aucune supervision. Si elle ressemble à du bruit, le problème vient des
embeddings, pas du k-means.

In [ ]:
from sklearn.decomposition import PCA

flat = res.patch_embedding.astype(np.float32)
flat = flat / (np.linalg.norm(flat, axis=1, keepdims=True) + 1e-8)
pca = PCA(n_components=3, random_state=0).fit(flat)
proj = pca.transform(flat).reshape(res.cell_id.size, G, G, 3)

lo, hi = np.percentile(proj, [2, 98])
pca_rgb = np.clip((proj - lo) / (hi - lo), 0, 1)
print("variance expliquée par les 3 premières composantes :",
      np.round(pca.explained_variance_ratio_, 3))

n = min(6, res.cell_id.size)
fig, axes = plt.subplots(2, n, figsize=(2.4 * n, 5.2), squeeze=False)
for j in range(n):
    axes[0, j].imshow(to_rgb(kept_tiles[j])); axes[0, j].set_axis_off()
    axes[0, j].set_title(f"RGB  {res.cell_id[j]}", fontsize=8)
    axes[1, j].imshow(pca_rgb[j], interpolation="nearest"); axes[1, j].set_axis_off()
    axes[1, j].set_title("PCA des tokens", fontsize=8)
fig.tight_layout(); plt.show()

## 6. k-means — **la cellule à rejouer pour régler `K`**

Rien de coûteux ici : on peut relancer cette cellule autant de fois qu'on veut
en changeant `K` (défini en §1, ou écrasé juste en dessous).

Le tableau imprimé donne, pour chaque groupe, sa taille et sa **couleur moyenne
en RGB** — c'est ce qui permet de dire « ce cluster, c'est de la forêt » sans
étiquettes.

In [ ]:
from sklearn.cluster import KMeans

K = 12          # <-- à modifier et réexécuter

z = flat        # déjà normalisé L2
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(z)
lab = km.labels_.reshape(res.cell_id.size, G, G)
cmap = categorical_cmap(K)

# couleur moyenne (réflectance) de chaque groupe : moyenne des 16x16 px du patch
S = kept_tiles.shape[-1]
px = kept_tiles.reshape(res.cell_id.size, 3, G, S // G, G, S // G).mean(axis=(3, 5))
px = np.transpose(px, (0, 2, 3, 1)).reshape(-1, 3)          # [patch, 3]
print(f" k | taille  |   R     V     B   (réflectance moyenne)")
for k in range(K):
    m = km.labels_ == k
    r, g, b = np.nanmean(px[m], axis=0)
    print(f"{k:2d} | {m.sum():6d}  | {r:.3f} {g:.3f} {b:.3f}")

sil = None
try:
    from sklearn.metrics import silhouette_score
    sub = np.random.default_rng(0).choice(z.shape[0], size=min(5000, z.shape[0]), replace=False)
    sil = silhouette_score(z[sub], km.labels_[sub], metric="cosine")
    print(f"\nsilhouette (cosinus, échantillon) : {sil:.3f}")
except Exception as e:
    print("silhouette non calculée :", e)

In [ ]:
n = min(6, res.cell_id.size)
fig, axes = plt.subplots(3, n, figsize=(2.4 * n, 7.6), squeeze=False)
for j in range(n):
    axes[0, j].imshow(to_rgb(kept_tiles[j])); axes[0, j].set_title(f"RGB {res.cell_id[j]}", fontsize=8)
    axes[1, j].imshow(pca_rgb[j], interpolation="nearest"); axes[1, j].set_title("PCA", fontsize=8)
    axes[2, j].imshow(lab[j], cmap=cmap, vmin=-0.5, vmax=K - 0.5, interpolation="nearest")
    axes[2, j].set_title(f"k-means K={K}", fontsize=8)
    for a in axes[:, j]:
        a.set_axis_off()
fig.tight_layout(); plt.show()

Comment lire ces trois lignes :

- si **PCA** montre les structures mais que **k-means** les hache, `K` est trop
  grand (ou les groupes se partagent une même texture) ;
- si k-means colle des objets visiblement différents dans le même groupe, `K`
  est trop petit ;
- si k-means suit un dégradé de luminosité plutôt que les objets, les
  embeddings sont dominés par l'éclairement : essayer `pooling`/normalisation,
  ou une date moins nuageuse.

### 6b. À quoi ressemble un groupe donné ?

Extrait les patches de 16 px les plus proches du centre du groupe choisi.

In [ ]:
CLUSTER = 0     # <-- groupe à inspecter

d = np.linalg.norm(z - km.cluster_centers_[CLUSTER], axis=1)
d[km.labels_ != CLUSTER] = np.inf
best = np.argsort(d)[:24]

p = S // G                                  # côté d'un patch en pixels
ti, ri, ci = np.unravel_index(best, (res.cell_id.size, G, G))
patches = [to_rgb(kept_tiles[t][:, r * p:(r + 1) * p, c * p:(c + 1) * p], 5, 95)
           for t, r, c in zip(ti, ri, ci)]
show_tiles(patches, [f"tuile {res.cell_id[t]}" for t in ti], ncol=8, size=1.5)
plt.suptitle(f"groupe {CLUSTER} — {(km.labels_ == CLUSTER).sum()} patches", y=1.02)
plt.show()

## 7. Carte HEALPix des groupes

Avec `healpix_plot`, en lon/lat. Les patches sont au niveau `res.patch_level`.

In [ ]:
import cartopy.crs as ccrs
import healpix_plot

grid_px    = healpix_plot.HealpixGrid(level=level, indexing_scheme="nested", ellipsoid="WGS84")
grid_patch = healpix_plot.HealpixGrid(level=res.patch_level, indexing_scheme="nested",
                                      ellipsoid="WGS84")

fig, axes = plt.subplots(1, 2, figsize=(15, 6),
                         subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
hi = np.nanpercentile(rgb, 98)
healpix_plot.plot(cell_id, rgb, healpix_grid=grid_px, sampling_grid={"shape": 900},
                  ax=axes[0], rgb_clip=(0.0, float(hi)), axis_labels="none",
                  title=f"{dates[TIME_INDEX]}  RGB (niveau {level})")
mp = healpix_plot.plot(res.patch_cell_id, km.labels_.astype(np.float32),
                       healpix_grid=grid_patch, sampling_grid={"shape": 900},
                       ax=axes[1], cmap=cmap, vmin=-0.5, vmax=K - 0.5, axis_labels="none",
                       title=f"k-means K={K} (niveau {res.patch_level})")
fig.colorbar(mp, ax=axes[1], shrink=0.7, ticks=range(K))
plt.show()

## 8. UMAP des embeddings

Uniquement pour voir la structure de l'espace : des groupes bien séparés ici
veulent dire que le k-means a un sens. UMAP ne sert pas à classer.

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                    metric="cosine", random_state=0)
u = reducer.fit_transform(z)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].scatter(u[:, 0], u[:, 1], c=km.labels_, cmap=cmap, s=2, alpha=0.6,
                vmin=-0.5, vmax=K - 0.5)
axes[0].set_title(f"UMAP, couleur = groupe k-means (K={K})")
axes[1].scatter(u[:, 0], u[:, 1], c=np.clip(px / max(np.nanpercentile(px, 98), 1e-6), 0, 1),
                s=2, alpha=0.8)
axes[1].set_title("UMAP, couleur = couleur RGB réelle du patch")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
plt.show()

Le panneau de droite est le plus parlant : si les couleurs réelles y forment un
dégradé continu alors que les groupes k-means découpent ce dégradé en tranches,
c'est que les embeddings encodent surtout la couleur moyenne — donc DINOv3
n'apporte rien de plus qu'un seuillage RGB, et il faut regarder du côté de la
normalisation ou de la taille des tuiles.

## 9. Comparaison utile : DINOv3 contre un k-means sur la couleur seule

Si les deux cartes se ressemblent, l'embedding n'apporte pas de texture.

In [ ]:
km_rgb = KMeans(n_clusters=K, n_init=10, random_state=0).fit(np.nan_to_num(px))
lab_rgb = km_rgb.labels_.reshape(res.cell_id.size, G, G)

from sklearn.metrics import adjusted_rand_score
print("indice de Rand ajusté entre les deux partitions : "
      f"{adjusted_rand_score(km.labels_, km_rgb.labels_):.3f}  "
      "(1 = identiques, 0 = sans rapport)")

n = min(5, res.cell_id.size)
fig, axes = plt.subplots(3, n, figsize=(2.4 * n, 7.6), squeeze=False)
for j in range(n):
    axes[0, j].imshow(to_rgb(kept_tiles[j])); axes[0, j].set_title("RGB", fontsize=8)
    axes[1, j].imshow(lab[j], cmap=cmap, vmin=-0.5, vmax=K - 0.5, interpolation="nearest")
    axes[1, j].set_title("DINOv3", fontsize=8)
    axes[2, j].imshow(lab_rgb[j], cmap=cmap, vmin=-0.5, vmax=K - 0.5, interpolation="nearest")
    axes[2, j].set_title("couleur seule", fontsize=8)
    for a in axes[:, j]:
        a.set_axis_off()
fig.tight_layout(); plt.show()

## 10. Aide-mémoire de réglage

| symptôme | paramètre | effet |
|---|---|---|
| trop peu de groupes, tout est mélangé | `K` (§6) | rejouer §6 seulement |
| tuiles trop petites, pas de contexte | `TILE_LEVELS` | 8 → 256 px, 10 → 1024 px ; **reprendre à §3** |
| presque aucune tuile gardée | `MIN_COVERAGE` | 0.95 tolère quelques nuages ; reprendre à §3 |
| patches trop grossiers (160 m au niveau 19) | rien à régler | c'est 16 px, fixé par DINOv3 ; changer `level` en amont |
| carte bruitée, groupes sans cohérence spatiale | voir §5 | si la PCA est déjà du bruit, le problème est les embeddings |
| résultat identique à la couleur | §9 | augmenter `TILE_LEVELS`, vérifier la normalisation SAT |

Deux constantes valent la peine d'être vérifiées contre le *model card* de la
version des poids téléchargée : `SAT493M_MEAN` et `SAT493M_STD` dans
`healpix_analyse/dino.py`. Une normalisation fausse dégrade les features sans
produire d'erreur visible.

Une fois `K` et `TILE_LEVELS` réglés ici, on passe aux 88 dates avec
`dino_umap_sentinel2.py --clusters K --tile-levels ... --cache ~/s2_cache`.